In [0]:
----------------------------------------------------
-- Combining the viewership table witht the user_profile table
----------------------------------------------------

----------------------------------------------------
-- Viewership table
----------------------------------------------------

WITH viewership AS (
SELECT
    COALESCE (UserID0, userid4,0) AS userid,

    --Dates
    TO_CHAR(TO_DATE(RecordDate2), 'yyyymm') AS Month_id, -- TO_CHAR: convert a date into a string & TO_DATE(): Convert a string into a date
    To_DATE(RecordDate2) AS watch_date, -- is to extract the date from the timestamp in the table
    DAYNAME(RecordDate2) AS day_name,

    CASE
        WHEN day_name IN ('Sat', 'Sun') THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_class,

    MONTHNAME(RecordDate2) AS month_name,

    TO_CHAR(RecordDate2, 'dd') AS day_of_week,

    -- Time
    DATE_FORMAT(RecordDate2, 'HH:mm:ss') AS watch_time,
    
    HOUR(RecordDate2) AS hour_of_day,

    CASE
        WHEN watch_time BETWEEN '00:00:00' AND '05:59:59' THEN 'Midnight'
        WHEN watch_time BETWEEN '06:00:00' AND '11:59:59' THEN 'Morning'
        WHEN watch_time BETWEEN '12:00:00' AND '16:59:59' THEN 'Afternoon'
        WHEN watch_time BETWEEN '17:00:00' AND '23:59:59' THEN 'Evening'
    END AS time_of_day,

    DATE_FORMAT(`Duration 2`, 'HH:mm:ss') AS Duration,

    CASE
        WHEN Duration BETWEEN '00:01:00' AND '00:30:00' THEN 'Low Usage'
        WHEN Duration BETWEEN '00:30:01' AND '00:59:59' THEN 'Medium Usage'
        WHEN Duration > '00:59:59' THEN 'Hihg Usage'
    END AS Screen_time,

    CASE 
        WHEN Duration <= '00:02:00' THEN 'inactive_user'
        ELSE 'active_user'
    END AS User_flag,

    CASE 
        WHEN Channel2 IN ('SawSee', 'Sawsee') THEN 'SawSee'
        WHEN Channel2 IN ('Supersport Live Events', 'Live on SuperSport', 'SuperSport Live Events', 'DStv Events 1') THEN 'Live Events'
        WHEN Channel2 = 'E! Entertainment' THEN 'Entertainment'
        ELSE Channel2
    END AS TV_Channel,

    CASE
        WHEN COALESCE(UserID0, userid4) IS NOT NULL THEN 1
        ELSE 0
    END AS active_subscriber_flag

 FROM brightlearn.bright_tv.viewership
),

------------------------------------------------------------------------------
-- User_profile table
------------------------------------------------------------------------------
user_profile AS (
SELECT UserID,
    CASE
        WHEN gender = ' ' THEN 'Unknown'
        WHEN gender = 'None' THEN 'Unknown'
        ELSE gender
    END AS Gender_gp,

    CASE
        WHEN Race = 'other' THEN 'None'
             WHEN Race = ' ' THEN 'None'
        ELSE Race
    END AS Race_New,

    CASE
        WHEN Province = ' ' THEN 'Uncategorized'
        WHEN Province = 'None' THEN 'Uncategorized'
        ELSE Province
    END AS Region,

    CASE
        WHEN age BETWEEN 0 AND 4 THEN 'Infant'
        WHEN age BETWEEN 5 AND 12 THEN 'Child'
        WHEN age BETWEEN 13 AND 18 THEN 'Teenager'
        WHEN age BETWEEN 19 AND 35 THEN 'Youth'
        WHEN age BETWEEN 36 AND 59 THEN 'Adult'
        WHEN age BETWEEN 60 AND 74 THEN 'Senior'
        WHEN age >= 75 THEN 'Elderly'
    END AS Age_group,

    CASE
        WHEN (Email IS NOT NULL) AND (email != ' ') AND (email != 'None') THEN 1
        ELSE 0
    END AS Email_flag,

    CASE
        WHEN (`Social Media Handle` IS NOT NULL) AND (`Social Media Handle` != ' ') AND (`Social Media Handle` != 'None') THEN 1
        ELSE 0
    END AS social_media_flag

FROM `brightlearn`.`bright_tv`.`user_profile`
)

-----------------------------------------------------------------------------------------------------------------
-- Using a left Join to combined the tables
----------------------------------------------------------------------------------------------------------------
SELECT COALESCE (v.Userid, u.Userid) AS UserID,
        day_name,
        day_class,
        month_name,
        watch_time,
        hour_of_day,
        Duration,
        Screen_time,
        TV_Channel,
        Gender_gp,
        Race_New,
        Region,
        Age_group,
        Email_flag,
        social_media_flag,
        User_flag

FROM viewership AS v
LEFT JOIN user_profile AS u
ON v.userid = u.UserID
GROUP BY ALL
;


Databricks visualization. Run in Databricks to view.